# Requirement 8: Restricted access

> The interpreter must not be directly exposed to the public internet. Restrict inbound access to approved systems. Private connectivity from on-premises is not required.

**Covered by:** Sandbox

**How it is fulfilled:** The data plane accepts only Entra-authenticated callers with the Data Owner role, there are no access keys, and sandboxes expose no ports unless asked to.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
sandbox, session_id = new_session()

In [3]:
url = f"{DATA_GROUP_URL}/sandboxes?{DATA_API}"
r = requests.get(url, timeout=30)
check(f"anonymous caller -> {r.status_code}", r.status_code in (401, 403))
r = requests.get(url, headers=token("admin", "data"), timeout=30)
check(f"authenticated but not approved (admin) -> {r.status_code}", r.status_code == 403)
r = requests.get(url, headers=token("orchestrator", "data"), timeout=30)
check(f"approved orchestrator -> {r.status_code}", r.status_code == 200)

PASS  anonymous caller -> 401


PASS  authenticated but not approved (admin) -> 403


PASS  approved orchestrator -> 200


In [4]:
check("the session exposes no ports", not sandbox.get().ports)

PASS  the session exposes no ports


In [5]:
cleanup(sandbox)

**Caveat:** The data plane has no documented private endpoint, so it is reachable from the internet to authenticated callers. The requirement was relaxed accordingly; add conditional access on the orchestrator for network restrictions.